# Cloud-Native Geospatial Workflow

This notebook demonstrates using Tissot to validate and optimize data
for cloud-native geospatial workflows.

Cloud-native formats like FlatGeobuf and GeoParquet enable efficient
HTTP range-request access. Tissot's cloud checker domain validates
best practices for these formats.

In [ ]:
import json
import subprocess
from pathlib import Path

def tissot(command: str, file: str, **kwargs) -> dict:
    cmd = ["tissot", command, file, "--json"]
    for key, value in kwargs.items():
        if isinstance(value, bool) and value:
            cmd.append(f"--{key}")
        elif not isinstance(value, bool):
            cmd.extend([f"--{key}", str(value)])
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return json.loads(result.stdout)

## Step 1: Audit Current Format

Check cloud-native compliance of existing data.

In [ ]:
cloud_check = tissot("check", "../datasets/kentucky_roads.geojson", domain="cloud")

print(f"Cloud-native findings: {cloud_check['summary']['total']}")
for f in cloud_check['findings']:
    print(f"  [{f['severity']}] {f['rule_id']}")
    print(f"    {f['message']}")
    if f.get('suggestion'):
        print(f"    Suggestion: {f['suggestion']}")

## Step 2: Full Quality Assessment

Get a comprehensive score including cloud readiness.

In [ ]:
score = tissot("score", "../datasets/kentucky_roads.geojson")

print(f"Overall: {score['overall_score']}/100 ({score['grade']})")
print("\nCategory breakdown:")
for name, cat in score.get('categories', {}).items():
    cat_score = cat['score'] if isinstance(cat, dict) else cat
    print(f"  {name}: {cat_score}/100")

## Step 3: Batch Audit

Audit all files in a directory.

In [ ]:
data_dir = Path("../datasets")
extensions = {".geojson", ".gpkg", ".shp", ".fgb"}

results = []
for path in sorted(data_dir.glob("*")):
    if path.suffix.lower() in extensions:
        try:
            report = tissot("check", str(path), domain="cloud")
            warnings = report['summary'].get('warnings', 0)
            status = 'PASS' if warnings == 0 else 'WARN'
            results.append((path.name, status, report['summary']['total']))
            print(f"{status} {path.name}: {report['summary']['total']} findings")
        except Exception as e:
            print(f"ERROR {path.name}: {e}")

passing = sum(1 for _, s, _ in results if s == 'PASS')
print(f"\nPassing: {passing}/{len(results)} files")

## Cloud-Native Format Guide

| Format | Cloud-Optimized | Spatial Index | Best For |
|--------|----------------|---------------|----------|
| GeoJSON | No | No | Small datasets, APIs |
| Shapefile | No | .shx only | Legacy compatibility |
| FlatGeobuf | Yes | Built-in | Vector data, streaming |
| GeoParquet | Yes | Built-in | Analytics, large datasets |
| GeoPackage | Partial | SQLite R-Tree | Desktop GIS |

See the [Cloud Native Geo Formats Guide](https://guide.cloudnativegeo.org/) for more details.